In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from netgraph import Graph
from adjustText import adjust_text
import re

In [ ]:
df = pd.read_csv('../../data/NBT20FRI-new.csv')
df

In [ ]:
df['Line'].unique()

In [ ]:
df["From Station"].unique()

In [ ]:
def convert_to_overground(line):
    if line.startswith('LO '):
        return 'Overground'
    return line

In [ ]:
# Function to remove suffixes from station names
def remove_suffix(station):
    suffixes = [' LU', ' LO', ' EL', ' NR', ' TfL', ' Trams', ' DLR']
    for suffix in suffixes:
        if station.endswith(suffix):
            return station[:-len(suffix)]
    return station

In [ ]:
df['Line'] = df['Line'].apply(convert_to_overground)
df['From Station'] = df['From Station'].apply(remove_suffix)
df['To Station'] = df['To Station'].apply(remove_suffix)

In [ ]:
df

In [ ]:
df.to_csv('../../data/dataset20.csv', sep=',', encoding='utf-8', index=False, header=True)

In [ ]:
coordinates_df = pd.read_csv('../../data/stations-with-coordinates.csv')
coordinates_df

In [ ]:
G = nx.Graph()

# Add edges to the graph
for _, row in df.iterrows():
    G.add_edge(row['From Station'], row['To Station'], line=row['Line'])

In [ ]:
color_map = {
    'Bakerloo': 'brown',
    'Central': 'red',
    'District': 'green',
    'DLR': 'turquoise',
    'Elizabeth Line': 'purple',
    'H&C and Circle': 'yellow',
    'Jubilee': 'grey',
    'Metropolitan': 'magenta',
    'Northern': 'black',
    'Piccadilly': 'blue',
    'Victoria': 'lightblue',
    'Waterloo & City': 'cyan',
    'Overground': 'orange',
    'London Trams': 'lime'
}

edge_colors = [color_map.get(G[u][v]['line'], 'gray') for u, v in G.edges()]

In [ ]:
def get_node_color(node):
    lines = set(G[node][neighbor]['line'] for neighbor in G[node])
    if len(lines) == 1:
        return color_map[list(lines)[0]]
    else:
        # If multiple lines, you could implement a priority system here
        # For now, we'll just take the first line's color
        return color_map[list(lines)[0]]

In [ ]:
# Get colors for all nodes
node_colors = [get_node_color(node) for node in G.nodes()]

## cleaning the dataframes

In [ ]:
def clean_station_name(name):
    """Clean and standardize station names."""
    name = re.sub(r'\s*\([^)]*\)', '', name)  # Remove text within parentheses
    name = re.sub(r'[^\w\s]', '', name)  # Remove special characters
    name = name.lower()  # Convert to lowercase
    return name

def merge_datasets(df, coordinates_df):
    # Clean station names in both dataframes
    df['From Station Cleaned'] = df['From Station'].apply(clean_station_name)
    df['To Station Cleaned'] = df['To Station'].apply(clean_station_name)
    coordinates_df['Name Cleaned'] = coordinates_df['Name'].apply(clean_station_name)

    # Merge the dataframes based on the cleaned station names
    merged_from = pd.merge(df, coordinates_df, how='left', left_on='From Station Cleaned', right_on='Name Cleaned', suffixes=('', '_From'))
    merged_to = pd.merge(df, coordinates_df, how='left', left_on='To Station Cleaned', right_on='Name Cleaned', suffixes=('', '_To'))

    # Collect missing stations with an indication of their source
    missing_from = merged_from[merged_from['Name'].isna()][['From Station']].drop_duplicates()
    missing_from['Missing From'] = 'From Station'

    missing_to = merged_to[merged_to['Name'].isna()][['To Station']].drop_duplicates()
    missing_to['Missing From'] = 'To Station'

    # Combine missing stations into one dataframe
    missing_stations = pd.concat([missing_from.rename(columns={'From Station': 'Station'}), 
                                  missing_to.rename(columns={'To Station': 'Station'})]).drop_duplicates().reset_index(drop=True)

    # Combine the coordinates from both merges
    merged_df = pd.concat([merged_from[['Line', 'Dir', 'From Station', 'To Station', 'Latitude', 'Longitude']], 
                           merged_to[['Latitude', 'Longitude']]], axis=1)

    return merged_df, missing_stations

In [ ]:
merged_df, missing_stations = merge_datasets(df, coordinates_df)

# Print the results

print("\nMissing Stations:")
missing_stations.head(60)

## plotting with the old

In [ ]:
plt.figure(figsize=(40, 20))
pos = nx.kamada_kawai_layout(G)
pos = {node: (-x, y) for node, (x, y) in pos.items()}  # Mirror horizontally

nx.draw(G, pos, edge_color=edge_colors, node_size=30, node_color=node_colors, with_labels=False)

# Create texts for labels
texts = []
for node, (x, y) in pos.items():
    texts.append(plt.text(x, y, node, fontsize=8))

# Adjust label positions to reduce overlap
adjust_text(texts)

plt.title("London TfL Network (NetworkX)")
plt.axis('off')
plt.show()

In [ ]:
plt.figure(figsize=(40, 20))  # Adjusted for better visibility
edge_color_dict = {edge: color_map.get(G.edges[edge]['line'], 'gray') for edge in G.edges()}
Graph(G, 
      edge_color=edge_color_dict, 
      node_size=0.1,  # Increased node size
      node_color="lightblue",  # Changed node color for visibility
      edge_width=0.5,  # Increased edge width
      node_labels=True,  # Removed node labels for clarity
      edge_layout='curved',  # Added curved edges for better distinction
      node_layout='radial',  # Uses spring layout for node positioning
      )
plt.title("London TfL Network (Netgraph)", fontsize=10)
plt.axis('off')
plt.show()